# 03 — Ablation study

Measures how much each stage of the pipeline contributes (Sections 6 and 16):
re-runs the ranking with individual features zeroed, and compares plain top-k
against MMR diversification on top-1 agreement, MRR and tone diversity.

In [ ]:
import sys, pathlib, numpy as np
sys.path.insert(0, str(pathlib.Path('..', 'backend').resolve()))

from app.ai.scoring import DEFAULT_WEIGHTS, FEATURE_NAMES, scores
from app.ai.mmr import mmr_select

# Synthetic feature matrix: 6 candidates x 6 features (Section 5.3 order)
F = np.array([
    [0.91, 0.84, 0.72, 0.63, 0.88, 0.55],
    [0.86, 0.79, 0.66, 0.71, 0.74, 0.62],
    [0.78, 0.90, 0.58, 0.55, 0.81, 0.49],
    [0.74, 0.68, 0.81, 0.66, 0.69, 0.71],
    [0.69, 0.75, 0.64, 0.80, 0.63, 0.58],
    [0.61, 0.70, 0.55, 0.58, 0.77, 0.66],
])
full = scores(F, DEFAULT_WEIGHTS)
print("full model :", np.round(full, 4))

for i, name in enumerate(FEATURE_NAMES):
    w = DEFAULT_WEIGHTS.copy()
    w[i] = 0.0
    w = w / w.sum()
    print(f"without {name:<16}:", np.round(scores(F, w), 4))

In [ ]:
# Plain top-k vs MMR diversification (Section 6.6)
rng = np.random.default_rng(0)
E = rng.normal(size=(6, 384))
E /= np.linalg.norm(E, axis=1, keepdims=True)

print("top-3 by score :", np.argsort(-full)[:3].tolist())
print("MMR top-3      :", mmr_select(full, E, k=3, lam=0.7))